In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from keras.models import Sequential, Model
from keras.losses import MeanSquaredError
from keras.metrics import RootMeanSquaredError
from keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import tensorflow.keras.backend as K
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, AveragePooling1D, LSTM, BatchNormalization, Attention, Dropout, Dense, GlobalAveragePooling1D, Embedding, Reshape, Concatenate

In [6]:
file_paths = ["/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2012_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2013_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2014_ kWh.csv", 
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2015_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2016_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2020_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2021_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2022_ kWh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2023_ kwh.csv",
              "/Users/iraklisdimitriadis/Documents/Μαθήματα-Σημειώσεις Data Science/Dissertation/Raw Data/2024_ kwh.csv"]

In [7]:
dataframes = []

for file_path in file_paths:
    df = pd.read_csv(file_path, delimiter=',')
    
    df[['Date', 'Kwh']] = df['Time'].str.split(',', expand=True)
    
    df = df[['Date', 'Kwh']]
    
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)

combined_df.set_index('Date', inplace=True)

print(combined_df.head())

                       Kwh
Date                      
23/02/2012 07:00  "0.0063"
23/02/2012 08:00  "0.1181"
23/02/2012 09:00  "0.3531"
23/02/2012 10:00   "0.445"
23/02/2012 11:00  "0.6291"


In [8]:
combined_df['Kwh'] = combined_df['Kwh'].str.replace('\"', '', regex=False)

In [9]:
combined_df.index = pd.to_datetime(combined_df.index, format='%d/%m/%Y %H:%M' )
combined_df['Kwh'] = pd.to_numeric(combined_df["Kwh"])

In [10]:
df_past = combined_df[combined_df.index < "2020-01-01 00:00:00"]
df_new = combined_df[combined_df.index >= "2020-01-01 00:00:00"]

In [11]:
full_index_past = pd.date_range(start=df_past.index.min(), end=df_past.index.max(), freq="H")
full_index_new = pd.date_range(start= df_new.index.min(), end = df_new.index.max(), freq= "H")


In [12]:
df_past = df_past.reindex(full_index_past, fill_value=0)
df_new = df_new.reindex(full_index_new, fill_value= 0)
df_past, df_new

(                        Kwh
 2012-02-23 07:00:00  0.0063
 2012-02-23 08:00:00  0.1181
 2012-02-23 09:00:00  0.3531
 2012-02-23 10:00:00  0.4450
 2012-02-23 11:00:00  0.6291
 ...                     ...
 2016-12-30 13:00:00  0.3201
 2016-12-30 14:00:00  0.3092
 2016-12-30 15:00:00  0.1746
 2016-12-30 16:00:00  0.0659
 2016-12-30 17:00:00  0.0004
 
 [42539 rows x 1 columns],
                         Kwh
 2020-01-01 08:00:00  0.0513
 2020-01-01 09:00:00  0.2104
 2020-01-01 10:00:00  0.3981
 2020-01-01 11:00:00  0.5015
 2020-01-01 12:00:00  0.5235
 ...                     ...
 2024-07-04 19:00:00  0.0592
 2024-07-04 20:00:00  0.0085
 2024-07-04 21:00:00  0.0000
 2024-07-04 22:00:00  0.0000
 2024-07-04 23:00:00  0.0000
 
 [39520 rows x 1 columns])

# Correlation Between Kwh - Temperature - Solar Radiation 

In [13]:
from sklearn.linear_model import LinearRegression

In [14]:
df_past

,Kwh
2012-02-23 07:00:00,0.0063
2012-02-23 08:00:00,0.1181
2012-02-23 09:00:00,0.3531
2012-02-23 10:00:00,0.4450
2012-02-23 11:00:00,0.6291
...,...
2016-12-30 13:00:00,0.3201
2016-12-30 14:00:00,0.3092
2016-12-30 15:00:00,0.1746
2016-12-30 16:00:00,0.0659
